# 05 Baselines: single-label and multi-label
Runs comparable baseline experiments and stores fold-level + summary results as CSV.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_recall_fscore_support, f1_score, accuracy_score, hamming_loss, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.multioutput import MultiOutputClassifier
try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None
try:
    from sentence_transformers import SentenceTransformer
except Exception:
    SentenceTransformer = None

DATA_DIR = Path('../data/biotools')
RESULTS_DIR = Path('../results'); RESULTS_DIR.mkdir(exist_ok=True, parents=True)
RANDOM_STATE = 42
N_SPLITS = 5
ATTRIBUTES = {
    'name': ['name'],
    'description': ['description'],
    'repo_title_keywords': ['repo_title','keywords'],
    'readme_description': ['readme','description'],
    'abstract': ['abstract'],
    'paper_title': ['paper_title'],
    'abstract_repo_metadata': ['abstract','repo_title','keywords'],
    'abstract_readme_description': ['abstract','readme','description'],
}


In [ ]:
def load_data():
    single = pd.read_csv(DATA_DIR/'biotools_singlelabel.csv').fillna('')
    multi = pd.read_csv(DATA_DIR/'biotools_multilabel.csv').fillna('')
    return single, multi

def combine_text(df, cols):
    cols=[c for c in cols if c in df.columns]
    return df[cols].astype(str).agg(' '.join, axis=1).str.replace(r'\s+', ' ', regex=True).str.strip()

def valid_attr(df, cols):
    text = combine_text(df, cols)
    return text.str.len() > 0

single, multi = load_data()
print(single.shape, multi.shape)
display(single.head(2))


## Single-label baselines

In [ ]:
def single_models():
    models = {
        'tfidf_logreg': Pipeline([('tfidf', TfidfVectorizer(max_features=50000, ngram_range=(1,2), min_df=2, sublinear_tf=True)), ('clf', LogisticRegression(max_iter=3000, class_weight='balanced', n_jobs=-1))]),
        'tfidf_linear_svm': Pipeline([('tfidf', TfidfVectorizer(max_features=50000, ngram_range=(1,2), min_df=2, sublinear_tf=True)), ('clf', LinearSVC(class_weight='balanced'))]),
        'tfidf_random_forest': Pipeline([('tfidf', TfidfVectorizer(max_features=30000, min_df=2, sublinear_tf=True)), ('clf', RandomForestClassifier(n_estimators=300, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1))]),
    }
    if XGBClassifier is not None:
        models['tfidf_xgboost'] = Pipeline([('tfidf', TfidfVectorizer(max_features=30000, min_df=2, sublinear_tf=True)), ('clf', XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, eval_metric='mlogloss', random_state=RANDOM_STATE, n_jobs=-1))])
    return models

def evaluate_single(df, attr_name, cols):
    mask = valid_attr(df, cols)
    sub = df[mask].copy()
    X = combine_text(sub, cols).values
    y = sub['primary_topic'].values
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    rows=[]; reports=[]
    for model_name, model in single_models().items():
        for fold,(tr,te) in enumerate(skf.split(X,y),1):
            model.fit(X[tr], y[tr])
            pred=model.predict(X[te])
            p,r,f,_=precision_recall_fscore_support(y[te], pred, average='macro', zero_division=0)
            rows.append({'task':'single','attribute':attr_name,'model':model_name,'fold':fold,'precision_macro':p,'recall_macro':r,'f1_macro':f,'accuracy':accuracy_score(y[te], pred),'n_train':len(tr),'n_test':len(te)})
            rep=pd.DataFrame(classification_report(y[te], pred, output_dict=True, zero_division=0)).T.reset_index().rename(columns={'index':'label'})
            rep['task']='single'; rep['attribute']=attr_name; rep['model']=model_name; rep['fold']=fold
            reports.append(rep)
    return pd.DataFrame(rows), pd.concat(reports, ignore_index=True)

all_rows=[]; all_reports=[]
for attr, cols in ATTRIBUTES.items():
    if any(c in single.columns for c in cols):
        r, rep = evaluate_single(single, attr, cols)
        all_rows.append(r); all_reports.append(rep)
        print(attr, r.groupby('model')['f1_macro'].mean().sort_values(ascending=False).head())
single_results=pd.concat(all_rows, ignore_index=True)
single_reports=pd.concat(all_reports, ignore_index=True)
single_results.to_csv(RESULTS_DIR/'biotools_single_baselines_fold_results.csv', index=False)
single_reports.to_csv(RESULTS_DIR/'biotools_single_per_class_results.csv', index=False)
single_summary=single_results.groupby(['task','attribute','model']).agg(['mean','std']).reset_index()
single_summary.to_csv(RESULTS_DIR/'biotools_single_baselines_summary.csv', index=False)
display(single_summary.head())


## Multi-label baselines

In [ ]:
def multilabel_models():
    base = {
        'tfidf_logreg_ovr': OneVsRestClassifier(Pipeline([('tfidf', TfidfVectorizer(max_features=50000, ngram_range=(1,2), min_df=2, sublinear_tf=True)), ('clf', LogisticRegression(max_iter=3000, class_weight='balanced'))])),
        'tfidf_linear_svm_ovr': OneVsRestClassifier(Pipeline([('tfidf', TfidfVectorizer(max_features=50000, ngram_range=(1,2), min_df=2, sublinear_tf=True)), ('clf', LinearSVC(class_weight='balanced'))])),
        'tfidf_random_forest_ovr': OneVsRestClassifier(Pipeline([('tfidf', TfidfVectorizer(max_features=30000, min_df=2, sublinear_tf=True)), ('clf', RandomForestClassifier(n_estimators=300, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1))])),
    }
    return base

def iterative_or_random_split(Y):
    try:
        from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
        return MultilabelStratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(np.zeros(len(Y)), Y)
    except Exception:
        return StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(np.zeros(len(Y)), Y.sum(axis=1))

def evaluate_multi(df, attr_name, cols):
    mask = valid_attr(df, cols) & df['topic_terms'].astype(str).str.len().gt(0)
    sub=df[mask].copy()
    X=combine_text(sub, cols).values
    labels=sub['topic_terms'].str.split('|').tolist()
    mlb=MultiLabelBinarizer(); Y=mlb.fit_transform(labels)
    rows=[]; reports=[]
    for model_name, model in multilabel_models().items():
        for fold,(tr,te) in enumerate(iterative_or_random_split(Y),1):
            model.fit(X[tr], Y[tr])
            pred=model.predict(X[te])
            rows.append({'task':'multi','attribute':attr_name,'model':model_name,'fold':fold,
                         'f1_micro':f1_score(Y[te], pred, average='micro', zero_division=0),
                         'f1_macro':f1_score(Y[te], pred, average='macro', zero_division=0),
                         'f1_weighted':f1_score(Y[te], pred, average='weighted', zero_division=0),
                         'subset_accuracy':accuracy_score(Y[te], pred), 'hamming_loss':hamming_loss(Y[te], pred),
                         'n_train':len(tr), 'n_test':len(te), 'n_labels':Y.shape[1]})
            rep=pd.DataFrame(classification_report(Y[te], pred, target_names=mlb.classes_, output_dict=True, zero_division=0)).T.reset_index().rename(columns={'index':'label'})
            rep['task']='multi'; rep['attribute']=attr_name; rep['model']=model_name; rep['fold']=fold
            reports.append(rep)
    return pd.DataFrame(rows), pd.concat(reports, ignore_index=True)

all_rows=[]; all_reports=[]
for attr, cols in ATTRIBUTES.items():
    if any(c in multi.columns for c in cols):
        r, rep = evaluate_multi(multi, attr, cols)
        all_rows.append(r); all_reports.append(rep)
        print(attr, r.groupby('model')['f1_macro'].mean().sort_values(ascending=False).head())
multi_results=pd.concat(all_rows, ignore_index=True)
multi_reports=pd.concat(all_reports, ignore_index=True)
multi_results.to_csv(RESULTS_DIR/'biotools_multi_baselines_fold_results.csv', index=False)
multi_reports.to_csv(RESULTS_DIR/'biotools_multi_per_class_results.csv', index=False)
multi_summary=multi_results.groupby(['task','attribute','model']).agg(['mean','std']).reset_index()
multi_summary.to_csv(RESULTS_DIR/'biotools_multi_baselines_summary.csv', index=False)
display(multi_summary.head())
